# Clustered frequencies: a basis the FFT cannot see

`CC-BY 2026 Brooksbank, Kassabov, Wilson`

A signal that is a sum of a few damped exponentials is sparse in a spectral basis, but not in the FFT's basis unless its frequencies happen to sit on the FFT grid and far apart.  When two frequencies are closer than one FFT bin, the FFT needs a longer record to separate them.  A short record of such a signal sampled on a grid is, however, a tensor of a very particular kind: a sum of a few *separable* exponentials,

$$T[i,j,k] = \sum_m c_m\, e^{(\!-\alpha_m + 2\pi i f^x_m) i}\, e^{(-\beta_m + 2\pi i f^y_m) j}\, e^{(-\gamma_m + 2\pi i f^z_m) k},$$

and Dleto's chisels find per-axis bases in which such a tensor has one nonzero per component.  Those bases are the exponentials themselves, at their true, non-uniform frequencies.  This notebook shows the mechanism on a synthetic record and then on measured NMR data, where chemical shifts cluster and the indirect dimensions are short because every increment costs scanner time.

**Method.** `nondeg` reduces the record to its $M\times M\times M$ core; the core has exactly $2M$ derivations (two of them trivial), so `der` and `stratify` are called with `nd = 2M`; `stratify` diagonalises the core into *strata* with $M$ nonzeros; the factor of stratified index $p$ on axis $a$ is column $p$ of $\overline{E_a}\,X_a^{-\top}$ ($E$ the nondeg basis, $X$ the stratify transform), and a weighted log-ratio fit reads its damping and frequency.  Success means **substantially fewer nonzeros in the strata than in the record**, with the recovered frequencies matching.

---
 1. [Setting up](#1.-Setting-up)
 2. [Theory: a synthetic record](#2.-Theory:-a-synthetic-record)
 3. [Real data, two measured axes: an HSQC of AMP](#3.-Real-data,-two-measured-axes:-an-HSQC-of-AMP)
 4. [Real data, three measured axes: an HNCO of a protein](#4.-Real-data,-three-measured-axes:-an-HNCO-of-a-protein)
 5. [What was and was not shown](#5.-What-was-and-was-not-shown)

## 1. Setting up

Run from the `labs/` folder of a clone of [OpenDleto](https://github.com/thetensor-space/OpenDleto).  The helper module `harmonic-retrieval/HarmonicRetrieval.jl` holds the Bruker reader, the band selection and the readout; it needs only packages already in the project.

In [ ]:
using Pkg
Pkg.activate("../")
using ITensors, Plots, LinearAlgebra, Statistics, Printf, HDF5, FFTW, Random
using Dleto
include("harmonic-retrieval/HarmonicRetrieval.jl"); using .HarmonicRetrieval

## 2. Theory: a synthetic record

Five damped exponentials on a $16\times16\times16$ grid.  Along each axis two pairs of frequencies are 0.02 cycles per sample apart, a third of the FFT's resolution of $1/16$.  Noise at 40 dB.

In [ ]:
n = 16
fx = [0.10, 0.12, 0.30, 0.32, 0.45];  fy = [0.21, 0.23, 0.07, 0.39, 0.395];  fz = [0.15, 0.33, 0.16, 0.05, 0.25]
Random.seed!(1)
T = synthetic_record(n, fx, fy, fz; amps = [1.0, 0.8, 0.9, 0.7, 1.0], snr_db = 40)

for (a, truth) in zip(("x", "y", "z"), (fx, fy, fz))
    println("FFT along $a finds ", length(fft_peaks_1d(T, a == "x" ? 1 : a == "y" ? 2 : 3)), " peaks; the truth has ", length(truth))
end

In [ ]:
T_norm = T / norm(T)
fr = [Index(size(T_norm, a), ("a", "b", "c")[a]) for a in 1:3]
Γ = ITensor(T_norm, fr...)
Δit, Es = nondeg(Γ; tol = 0.03)
ch = UniversalChisel(length(inds(Δit)))
Ω = IndTransverseOps(collect(inds(Δit)), UniversalOp())
ders = der(Ω, ch, Δit; nd = 10, tol = 0.1)

best = nothing
for _ in 1:8
    coefs = randn(length(ders))
    δ = [sum(coefs[j] * ders[j][a] for j in eachindex(ders)) for a in 1:3]
    St_try = stratify(Δit, δ)
    Σ_try = Array(St_try.Σ, inds(St_try.Σ)...)
    nz_try = count(x -> abs(x) > 0.1 * maximum(abs, Σ_try), Σ_try)
    if best === nothing || nz_try < best.nonzeros
        best = (; nonzeros = nz_try, St = St_try)
    end
end
St = best.St

Ems = Matrix{ComplexF64}[]
Xms = Matrix{ComplexF64}[]
e2s = Index[]
for a in 1:3
    E = only(filter(t -> hasind(t, fr[a]), Es))
    e2 = only(filter(i -> i != fr[a], inds(E)))
    X = only(filter(t -> hasind(t, e2), St.Xs))
    x2 = only(filter(i -> i != e2, inds(X)))
    push!(Ems, Array(E, fr[a], e2))
    push!(Xms, Array(X, e2, x2))
    push!(e2s, e2)
end

Δ = Array(Δit, inds(Δit)...)
Σ = Array(St.Σ, e2s...)
nonzeros = count(x -> abs(x) > 0.1 * maximum(abs, Σ), Σ)
record_nonzeros = count(x -> abs(x) > 0.1 * maximum(abs, T), T)
top = CartesianIndices(Σ)[partialsortperm(vec(abs2.(Σ)), 1:5; rev = true)]
factors = [[(conj(Ems[a]) * transpose(inv(Xms[a])))[:, Tuple(ci)[a]] for a in 1:3] for ci in top]

println("nondeg core ", size(Δ), ";  strata nonzeros ", nonzeros, " of ", length(Σ), "   (record: ", record_nonzeros, " of ", length(T), ")")
println("recovered frequencies (cycles/sample):")
for f in factors
    @printf("   x %.4f   y %.4f   z %.4f\n", frequency(f[1]), frequency(f[2]), frequency(f[3]))
end
println("true: ", collect(zip(fx, fy, fz)))

The record is dense; the strata hold five entries, one per component, and the frequencies come back to four decimals, clusters included.  (Drop the noise to see them exact, or go to 20 dB to see nondeg start to lose the weakest component.)

In [ ]:
plot(
    plot_tensor(abs.(T) ./ maximum(abs, T), 0.1;
        title = "record |T| $(join(size(T), "×"))",
        xlabel = "x", ylabel = "y", zlabel = "z", color = :blue),
    plot_tensor(abs.(Σ) ./ maximum(abs, Σ), 0.1;
        title = "strata |Σ| $(join(size(Σ), "×"))",
        xlabel = "a", ylabel = "b", zlabel = "c", color = :green);
    layout = (1, 2), size = (1100, 480))

## 3. Real data, two measured axes: an HSQC of AMP

A $^1$H–$^{13}$C HSQC of adenosine monophosphate (100 mM) from the BMRB metabolomics archive (entry bmse000005, Bruker 500 MHz, public).  It has 128 complex $^{13}$C increments over 170 ppm, so one FFT bin of the indirect axis is 1.3 ppm, and 1024 $^1$H points.  The next cell downloads the raw files (about 1 MB), removes the digital filter, combines the echo/antiecho pairs into a complex $t_1$, and picks the reference peaks from the full record.

In [ ]:
dir = download_bmrb_experiment("bmse000005", "set01", 6, joinpath("data", "bmse000005", "6"))
fid0, p = read_bruker_hsqc(dir)
fid = echo_antiecho(remove_digital_filter(fid0, p.grpdly))              # (t1 = 13C, t2 = 1H), complex
ppmC(f) = (p.o2 + f) / p.sfo1;  ppmH(f) = (p.o1 + f) / p.sfo2                # Hz offset from carrier -> ppm
n1, n2 = size(fid)
apod = [cos(π * i / (2n1)) for i in 0:n1-1] * transpose([cos(π * j / (2n2)) for j in 0:n2-1])
mag = abs.(fftshift(fft(fid .* apod)))
refs = reference_peaks(mag; thr = 0.08)[1:7]
println("$(p.pulprog), $(p.ns) scans; 13C: $(n1) increments, FFT bin $(round(p.sw1 / n1 / p.sfo1; digits = 2)) ppm")
println("reference cross peaks from the full record (13C ppm, 1H ppm):")
for (i, j, h) in refs; @printf("   %6.2f  %6.2f\n", ppmC((i - n1 ÷ 2 - 1) * p.sw1 / n1), ppmH((j - n2 ÷ 2 - 1) * p.sw2 / n2)); end

The ribose carbons C5′, C3′, C2′ and C4′ (63–85 ppm, with C5′ carrying two protons) sit under the $^1$H band 3.8–4.7 ppm.  We select that band by mixing the $^1$H axis to its centre and decimating it by 14, keep only the **first 16** $^{13}$C increments, and give Dleto a third mode by the Hankel construction $T[t_1, j, k] = \mathrm{fid}[t_1, j+k]$, which keeps every exponential separable.  At 16 increments the $^{13}$C FFT bin is 10.6 ppm; C3′ and C2′ are 4 ppm apart.

In [ ]:
sub = mix_decimate(fid, 2, p.sw2, -220.0, 14)        # 1H band centred 220 Hz upfield of the carrier (3.8-4.7 ppm)
N1 = 16
T2 = hankel_tensor(sub[1:N1, :], 16)                  # (t1, j, k) with j + k - 1 = decimated t2
T2_norm = T2 / norm(T2)
fr2 = [Index(size(T2_norm, a), ("a", "b", "c")[a]) for a in 1:3]
Γ2 = ITensor(T2_norm, fr2...)
Δ2it, E2 = nondeg(Γ2; tol = 0.05)
ch2 = UniversalChisel(length(inds(Δ2it)))
Ω2 = IndTransverseOps(collect(inds(Δ2it)), UniversalOp())
ders2 = der(Ω2, ch2, Δ2it; nd = 10, tol = 0.3)

best2 = nothing
for _ in 1:8
    coefs = randn(length(ders2))
    δ2 = [sum(coefs[j] * ders2[j][a] for j in eachindex(ders2)) for a in 1:3]
    St_try = stratify(Δ2it, δ2)
    Σ_try = Array(St_try.Σ, inds(St_try.Σ)...)
    nz_try = count(x -> abs(x) > 0.1 * maximum(abs, Σ_try), Σ_try)
    if best2 === nothing || nz_try < best2.nonzeros
        best2 = (; nonzeros = nz_try, St = St_try)
    end
end
St2 = best2.St

E2m = Matrix{ComplexF64}[]
X2m = Matrix{ComplexF64}[]
e22 = Index[]
for a in 1:3
    E = only(filter(t -> hasind(t, fr2[a]), E2))
    e2 = only(filter(i -> i != fr2[a], inds(E)))
    X = only(filter(t -> hasind(t, e2), St2.Xs))
    x2 = only(filter(i -> i != e2, inds(X)))
    push!(E2m, Array(E, fr2[a], e2))
    push!(X2m, Array(X, e2, x2))
    push!(e22, e2)
end

Δ2 = Array(Δ2it, inds(Δ2it)...)
Σ2 = Array(St2.Σ, e22...)
nonzeros2 = count(x -> abs(x) > 0.1 * maximum(abs, Σ2), Σ2)
record_nonzeros2 = count(x -> abs(x) > 0.1 * maximum(abs, T2), T2)
top2 = CartesianIndices(Σ2)[partialsortperm(vec(abs2.(Σ2)), 1:5; rev = true)]
factors2 = [[(conj(E2m[a]) * transpose(inv(X2m[a])))[:, Tuple(ci)[a]] for a in 1:3] for ci in top2]

println("13C increments used: $N1 (FFT bin $(round(p.sw1 / N1 / p.sfo1; digits = 1)) ppm);  nondeg core ", size(Δ2))
println("strata nonzeros ", nonzeros2, " of ", length(Σ2), "   (record: ", record_nonzeros2, " of ", length(T2), ")")
println("recovered cross peaks (13C ppm, 1H ppm):")
for f in factors2
    @printf("   %6.2f  %6.2f\n", ppmC(frequency(f[1]) * p.sw1), ppmH(frequency(f[3]) * p.sw2 / 14 - 220.0))
end

In [ ]:
plot(
    plot_tensor(abs.(T2) ./ maximum(abs, T2), 0.1;
        title = "record |T| $(join(size(T2), "×"))",
        xlabel = "13C t1", ylabel = "j", zlabel = "k", color = :blue),
    plot_tensor(abs.(Σ2) ./ maximum(abs, Σ2), 0.1;
        title = "strata |Σ| $(join(size(Σ2), "×"))",
        xlabel = "a", ylabel = "b", zlabel = "c", color = :green);
    layout = (1, 2), size = (1100, 480))

All four ribose carbons come back to within half a ppm (a third of the full record's own FFT bin) from 16 increments, including C3′ and C2′, which the 10.6 ppm bin cannot separate; the $^1$H shifts agree to a few hundredths of a ppm.  One mode here is manufactured, so Section 4 repeats the exercise where all three axes are measured.

## 4. Real data, three measured axes: an HNCO of a protein

A 3D HNCO of mouse thiamine triphosphatase (BMRB entry 15063, Varian 600 MHz) correlates each residue's amide $^1$H and $^{15}$N with the preceding carbonyl $^{13}$C′.  The record is 40 $^{15}$N × 32 $^{13}$C′ × 1024 $^1$H points; the carbonyl shifts of some 200 residues crowd into about 10 ppm while one $^{13}$C′ FFT bin is 0.62 ppm.  The file `harmonic-retrieval/data/bmr15063_hnco_band_3.56-3.68ppm.h5` is the $^1$H band 3.56–3.68 ppm (from the carrier) of that record, decimated along $^1$H only (see `harmonic-retrieval/data/README.md` for the preparation); it holds two strong cross peaks whose $^{13}$C′ shifts are 0.93 ppm apart, a weak third, and at its edge a fourth with $^{13}$C′ at −1.24 ppm.  No mode is constructed: the tensor is $(^{15}\!N, ^{13}\!C', ^1\!H)$ as acquired.  The nondeg tolerance is set to keep three components (the two strong peaks and the edge peak); the random derivation combination inside `stratify` is seeded so the cell reproduces.

In [ ]:
h = h5open("harmonic-retrieval/data/bmr15063_hnco_band_3.56-3.68ppm.h5", "r")
H = permutedims(read(h["fid"]), (3, 2, 1));  refs3 = permutedims(read(h["ref_peaks"]))
swN, swC, swH, obsN, obsC, obsH, fcH = (read_attribute(h, k) for k in ("swN", "swC", "swH", "obsN", "obsC", "obsH", "fcH"))
close(h)
println("record (15N, 13C', 1H) = ", size(H), ";  strong reference peaks (15N, 13C', 1H ppm from carrier):")
for s in eachrow(refs3[refs3[:, 4] .> 100, :]); @printf("   %6.2f  %6.2f  %6.3f   (height %.0f x floor)\n", s[1] / obsN, s[2] / obsC, s[3] / obsH, s[4]); end

In [ ]:
for NC in (32, 12)
    Random.seed!(1)
    T3 = H[:, 1:NC, :]
    T3_norm = T3 / norm(T3)
    fr3 = [Index(size(T3_norm, a), ("a", "b", "c")[a]) for a in 1:3]
    Γ3 = ITensor(T3_norm, fr3...)
    Δ3it, E3 = nondeg(Γ3; tol = 0.2)
    ch3 = UniversalChisel(length(inds(Δ3it)))
    Ω3 = IndTransverseOps(collect(inds(Δ3it)), UniversalOp())
    ders3 = der(Ω3, ch3, Δ3it; nd = 6, tol = 0.3)

    best3 = nothing
    for _ in 1:10
        coefs = randn(length(ders3))
        δ3 = [sum(coefs[j] * ders3[j][a] for j in eachindex(ders3)) for a in 1:3]
        St_try = stratify(Δ3it, δ3)
        Σ_try = Array(St_try.Σ, inds(St_try.Σ)...)
        nz_try = count(x -> abs(x) > 0.1 * maximum(abs, Σ_try), Σ_try)
        if best3 === nothing || nz_try < best3.nonzeros
            best3 = (; nonzeros = nz_try, St = St_try)
        end
    end
    St3 = best3.St

    E3m = Matrix{ComplexF64}[]
    X3m = Matrix{ComplexF64}[]
    e23 = Index[]
    for a in 1:3
        E = only(filter(t -> hasind(t, fr3[a]), E3))
        e2 = only(filter(i -> i != fr3[a], inds(E)))
        X = only(filter(t -> hasind(t, e2), St3.Xs))
        x2 = only(filter(i -> i != e2, inds(X)))
        push!(E3m, Array(E, fr3[a], e2))
        push!(X3m, Array(X, e2, x2))
        push!(e23, e2)
    end

    Δ3 = Array(Δ3it, inds(Δ3it)...)
    Σ3 = Array(St3.Σ, e23...)
    nonzeros3 = count(x -> abs(x) > 0.1 * maximum(abs, Σ3), Σ3)
    record_nonzeros3 = count(x -> abs(x) > 0.1 * maximum(abs, T3), T3)
    top3 = CartesianIndices(Σ3)[partialsortperm(vec(abs2.(Σ3)), 1:3; rev = true)]
    factors3 = [[(conj(E3m[a]) * transpose(inv(X3m[a])))[:, Tuple(ci)[a]] for a in 1:3] for ci in top3]

    println("\n13C' increments: $NC  (FFT bin $(round(swC / NC / obsC; digits = 2)) ppm);  nondeg core ", size(Δ3),
            ";  strata nonzeros ", nonzeros3, " of ", length(Σ3), "  (record: ", record_nonzeros3, " of ", length(T3), ")")
    for f in factors3
        @printf("   15N %6.2f   13C' %6.2f   1H %6.3f ppm\n", frequency(f[1]) * swN / obsN, frequency(f[2]) * swC / obsC, (frequency(f[3]) * swH + fcH) / obsH)
    end
    NC == 12 && display(plot(
        plot_tensor(abs.(T3) ./ maximum(abs, T3), 0.1;
            title = "record |T| $(join(size(T3), "×"))",
            xlabel = "15N", ylabel = "13C'", zlabel = "1H", color = :blue),
        plot_tensor(abs.(Σ3) ./ maximum(abs, Σ3), 0.1;
            title = "strata |Σ| $(join(size(Σ3), "×"))",
            xlabel = "a", ylabel = "b", zlabel = "c", color = :green);
        layout = (1, 2), size = (1100, 480)))
end

With 12 carbonyl increments the FFT bin is 1.66 ppm and the two strong peaks, 0.93 ppm apart (0.56 of a bin), come back at their reference $^{13}$C′ shifts to within 0.1 ppm, with $^{15}$N within 0.3 ppm and $^1$H within 0.01 ppm; the edge peak is found at $^{13}$C′ −1.2 ppm as well.  The strata hold three to five entries where the record has over a thousand.  This is reproducible across seeds down to 12 increments; at 8 increments (2.5 ppm bins) the pair collapses to its average, which is the limit for this record's signal-to-noise.  The weak third peak inside the band (171 × the noise floor, against 900 for the strong ones) is below what nondeg keeps at this tolerance.

## 5. What was and was not shown

* The mechanism is exact in theory and robust to noise on synthetic records: clustered frequencies a third of an FFT bin apart are recovered from 16 samples at 30–40 dB.
* On a measured 2D HSQC, with one mode built by the Hankel construction, all four ribose carbons of AMP are recovered from 16 of 128 increments, where the FFT cannot separate two of them.
* On a measured 3D HNCO, with all three modes acquired, the two strong carbonyl peaks of a band are resolved from 12 of 32 increments at 0.56 of an FFT bin, reproducibly; 8 increments is too few.  Weak peaks and crowded bands are not resolved: the limit is signal-to-noise per component and the number of components a short record can carry, not the FFT grid.
* Chemical shifts are reported relative to the carriers; nothing here is referenced to DSS.